# Direct Prompting (Qwen or OpenAI)

Run with **OpenAI API** (light, no local model) or **Qwen** (local). Use the cell below to choose, then type any prompt to get a reply.

In [6]:
# Choose backend (run this once)
USE_OPENAI = False  # True = OpenAI API (light). False = Qwen local (heavier).

import sys
from pathlib import Path

root = Path.cwd()
for _ in range(10):
    if (root / "dlp").is_dir():
        sys.path.insert(0, str(root))
        break
    root = root.parent

if USE_OPENAI:
    from dlp.models import OpenAILoader
    loader = OpenAILoader("gpt-4o-mini")  # cheap & fast; or "gpt-4o" for stronger
    loader.load()
    print("Using OpenAI API (set OPENAI_API_KEY in env). No local model.")
else:
    from dlp.models import HFLoader
    loader = HFLoader(
        "Qwen/Qwen2-1.5B-Instruct",
        device_map="cpu",
        local_files_only=False,
    )
    loader.load()
    print("Model ready.")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1374.12it/s, Materializing param=model.norm.weight]                              


Model ready.


## Your prompt

Edit the string below and run the cell to get the model's reply.

In [7]:
# Change this to your prompt
prompt = "List 5 unusual uses for a paperclip. Be super creative"

reply = loader.prompt(
    prompt,
    max_new_tokens=32,
    do_sample=True,
    temperature=0.7,
)

print("Reply:")
print(reply)

Reply:
Sure, here are five unusual uses for a paperclip:

1. As a makeshift fishing rod: Use the paperclip as a lightweight and portable fishing pole.


## Optional: system prompt

Use a system message to set the model's behavior, then ask your question.

In [8]:
system_prompt = "You are a creativity researcher. Be concise and varied."

user_prompt = "List 5 unusual uses for a rubber band."

reply = loader.prompt(
    user_prompt,
    system_prompt=system_prompt,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
)

print("Reply:")
print(reply)


Reply:
1. Creating a Tethering Device: Use multiple rubber bands to form a sturdy, flexible tether that can be used as a secure attachment or to provide a gentle hold.
2. Making a DIY Sock Hanger: Attach two rubber bands together to create a makeshift hanger, which can be used to hang socks without needing additional hardware.
3. Developing a DIY Lighter: Mix water and rubber bands to create a unique lighter by using the bands to seal an air pocket around the igniter and then blow on it to ignite the fuel inside.
4. Constructing a Stretchable Bandana: Fold rubber bands into different shapes to create stretchy headbands or face masks and use them as a fashion accessory.
5. Using Rubber Bands in Artistic Projects: Incorporate rubber bands into artistic designs such as painting, sculpture, or even crafting projects like making bracelets or belts from these materials.


In [10]:
import re

system_prompt ="""

You are an advanced reasoning engine designed to solve problems by finding non-obvious, creative solutions. To do this, you must avoid the default path (D) and instead discover a Bridge Mechanism (B) that transforms the problem space.

Your Task:

Analyze the Input: Read the provided problem or situation ($A$).

Propose a Bridge Mechanism: Before solving, explicitly define a transformation rule that changes how $A$ is represented. Your mechanism must fall into one of the following five families:

Re-representation: Reinterpret objects or constraints by their function or role to expose new affordances.

Structural mapping: Map $A$ to an isomorphic structure in another domain (analogy), solve there, then translate back.

Intermediate construct: Introduce a latent variable, auxiliary object, or subgoal that bridges the gap.

Method shift: Switch to a general reasoning principle (e.g., invariant, extremal argument, duality, symmetry).

Discourse/semantic operator: Apply a high-level rewrite (e.g., change Point of View, time frame, or resolve ambiguity) while preserving facts.

Validate the Mechanism: Your definition of $B$ must explicitly:

State the transformation.

Name the solution family it unlocks (what becomes possible that wasn't before).

Justify why the result remains appropriate and satisfies constraints.

Apply and Solve: Use mechanism $B$ to generate the final solution ($C$).

Input Problem:

[Insert Problem A Here]

Output Format:

Mechanism Family: [Select one from the list above]

Mechanism Definition: [State the transformation and the solution family it unlocks]

Justification: [Why is this valid?]

Final Solution: [The solution resulting from applying the mechanism]

"""

prompt = "List 5 unusual uses for a rubber band."
text = loader.prompt(
    prompt,
    system_prompt=system_prompt,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
).strip()

print(text)

Mechanism Family: Intermediate construct

Mechanism Definition: Introduce a latent variable, "band tension," to represent the stretchiness and flexibility of the rubber band.

Justification: The use of a latent variable allows us to introduce a novel variable that relates directly to the physical properties of the rubber band, enabling us to explore a range of unusual applications that exploit its inherent characteristics.

Final Solution: 
1. Creating a spring-like contraption using the rubber band's elastic properties.
2. Designing an inflatable dome shape with the rubber band's strength and elasticity.
3. Using the rubber band as a temporary suspension system for furniture items.
4. Generating a musical instrument with a rubber band's resonance and sustain characteristics.
5. Utilizing the rubber band's natural stretching and compressions for creating a unique form factor or aesthetic.


In [ ]:
prompt = '''
*** SYSTEM INSTRUCTION ***
You are an expert in lateral thinking and engineering. Your goal is to solve problems by overcoming "Functional Fixedness." You do not look at objects for what they *are* (e.g., "a spoon"), but for the abstract *mechanisms* they represent (e.g., "a concave lever").

*** THE PROTOCOL ***
For every object or problem presented, you must follow exactly three steps:

1.  **Step 1: Mechanism Extraction (The Bridge B)**
    Ignore the common use of the object. Isolate a specific physical property, scientific principle, or structural affordance. (e.g., Thermal mass, Tensile strength, Polarization, Capillary action).
    *Constraint:* Do not list common uses. Focus only on the abstract mechanism.

2.  **Step 2: Domain Shift**
    Find a completely different field (Architecture, Medicine, Music, Survival) where this specific mechanism is critical.

3.  **Step 3: Creative Instantiation (The Solution C)**
    Invent a novel product or solution using the object solely for that mechanism.
    *Constraint:* The solution must be specific and physically plausible.

*** EXAMPLES ***

Input Object: A Red Brick.
Output:
1.  **Mechanism (B):** High Thermal Mass (The ability to absorb, store, and slowly release heat energy).
2.  **Domain Shift:** Culinary / Cooking.
3.  **Creative Solution (C):** A "Primitive Sous-Vide Weight." Heat the brick to a specific stable temperature and wrap it in foil. Use it to press down on a marinated chicken on a grill, providing both weight for even searing and radiant heat from above for faster, even cooking.

Input Object: A Metal Paperclip.
Output:
1.  **Mechanism (B):** Electrical Conductivity (A malleable, uninsulated metal bridge).
2.  **Domain Shift:** Electronics / Hardware Hacking.
3.  **Creative Solution (C):** A "CMOS Reset Jumper." In an emergency where a computer fails to boot, bend the paperclip to short the two 'Clear CMOS' pins on the motherboard, resetting the BIOS settings when the standard jumper is missing.

*** YOUR TASK ***

Input Object: Rubber Band
Output:
'''
prompt="""*** SYSTEM INSTRUCTION ***
You are an expert in lateral thinking. Your goal is to find "Far Transfer" solutions by identifying obscure physical properties of common objects.

*** THE PROTOCOL ***

1.  **Step 1: Mechanism Extraction (The Bridge B)**
    Analyze the object. Identify a **SECONDARY or IGNORED** physical property or scientific principle.
    *CRITICAL CONSTRAINT:* You are FORBIDDEN from using the object's primary function.
    * For a Rubber Band: Do NOT use "Elasticity" or "Springiness". Do NOT suggest holding things together or shooting things.
    * Instead, look for: Friction, Flammability, Acoustics, Chemical composition, Thermal properties, Waterproofing.

2.  **Step 2: Domain Shift**
    Find a field completely unrelated to the object (e.g., Survival, Art, Plumbing, Electronics).

3.  **Step 3: Creative Instantiation (The Solution C)**
    Invent a novel product using ONLY that secondary mechanism.

*** YOUR TASK ***

Input Object: Rubber Band
Output:"""
prompt = """*** SYSTEM INSTRUCTION ***
You are a creative engine designed to bypass "Functional Fixedness". Your goal is to find the least obvious utility for common objects.

*** THE PROTOCOL ***

For the given input object, follow these 3 steps strictly:

Step 1: Divergent Analysis (List 3 Mechanisms)
Identify 3 distinct physical properties or mechanisms of the object.
* Mechanism A: The most obvious property (Primary Function).
* Mechanism B: A secondary property.
* Mechanism C: An obscure, ignored, or incidental property (The "Hidden Bridge").

Step 2: Selection & Inhibition
Explicitly REJECT Mechanisms A and B because they lead to boring solutions.
SELECT Mechanism C as your constraint.

Step 3: Creative Leap (The Solution)
Using ONLY Mechanism C, invent a novel solution or product.
State the domain you are shifting to.

*** EXAMPLE ***

Input: Candle.

Step 1:
* Mech A: It gives light (Illumination).
* Mech B: It generates heat (Thermodynamics).
* Mech C: The wax creates a waterproof, low-friction coating (Lubrication/Sealing).

Step 2:
* Rejecting A (Light) and B (Heat).
* Selecting C: Wax lubrication.

Step 3:
* Domain: Hardware Repair.
* Solution: A "Zipper Rescue Kit". Rub the unlit candle on the teeth of a stuck metal zipper. The wax creates a dry lubricant layer that unsticks the mechanism without staining the fabric like oil would.

*** YOUR TASK ***

Input Object: Rubber Band
Output:"""
text = loader.prompt(
    prompt,
    
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
).strip()

print(text)



Domain Shift: Sports Equipment Innovation

Mechanism A: It stretches rapidly in one direction.
Mechanism B: It contracts and releases quickly in response to tension.
Mechanism C: It can be used multiple times without losing its elasticity.

Step 1:
* Mech A: Rapid stretching.
* Mech B: Quick release.
* Mech C: Elasticity preservation.

Step 2:
* Rejecting A (Sparking) and B (Tension).
* Selecting C: Reuseable rubber bands.

Step 3:
* Domain Shift: Sports Equipment Innovation.
* Solution: Create an "Elastic Recycling System" where users can clip their unused elastic bands onto a machine that resew them together and repurpose them into new equipment, such as tennis balls or basketballs without needing to replace each band. This reduces waste and encourages recycling sports equipment among athletes.


In [19]:
import re

# --- פונקציות העזר לפרומפטים ---

def create_step1_prompt(object_name):
    """
    יוצר את הפרומפט לשלב הראשון: חילוץ מנגנון נסתר (B).
    אוסר על שימוש בתכונה הראשית.
    """
    return f"""
    [INST] 
    You are an expert in physics and material science.
    
    TASK: Analyze the object '{object_name}'.
    
    INSTRUCTIONS:
    1. Identify a SECONDARY or IGNORED physical property of this object.
    2. STRICTLY FORBIDDEN: Do not mention the object's primary use (e.g. if it's a rubber band, do not mention elasticity or holding things).
    3. STRICTLY FORBIDDEN: Do not mention the name '{object_name}' in your output.
    4. Focus on properties like: Friction, Chemical composition, Thermal mass, Acoustics, Flammability, Waterproofing.
    
    OUTPUT:
    Provide ONLY a short description of this physical property starting with "A material that...". Do not add introductions.
    [/INST]
    """
def create_step2_prompt(mechanism_b):
    return f"""
    [INST]
    You are a strictly logical engineer in a sci-fi setting.
    
    INPUT DATA:
    You have a material with this specific property:
    "{mechanism_b}"
    
    TASK:
    Invent a device that uses this property.
    
    CRITICAL INSTRUCTION:
    You must follow this 2-step thinking process explicitly:
    
    STEP 1: PHYSICAL IMPLICATION
    Explain clearly: What does this property allow us to do? 
    (e.g., If it resists heat -> we can use it in volcanoes. If it insulates electricity -> we can touch live wires).
    
    STEP 2: THE INVENTION
    Based *only* on the implication above, invent a survival tool.
    
    OUTPUT FORMAT:
    Analysis: [Your Step 1 reasoning]
    Invention Name: [Name]
    Mechanism: [How it works]
    [/INST]
    """
# --- הפונקציה הראשית להרצת הניסוי ---

def run_creative_bridge(loader, object_name):
    print(f"\n--- Processing: {object_name} ---")
    
    # 1. שלב א: חילוץ המנגנון (B)
    # משתמשים בטמפרטורה נמוכה יחסית כדי לקבל ניתוח פיזיקלי מדויק
    prompt_1 = create_step1_prompt(object_name)
    mechanism_b = loader.prompt(
        prompt_1,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.5, 
    ).strip()
    
    print(f"-> Extracted Mechanism (B): {mechanism_b}")
    
    # בדיקת תקינות: אם המודל פלט את שם החפץ, זה הורס את הניסוי
    if object_name.lower() in mechanism_b.lower():
        print("Warning: Model leaked the object name in Step 1. Retrying might be needed.")

    # 2. שלב ב: המצאה (C)
    # משתמשים בטמפרטורה גבוהה יותר ליצירתיות
    prompt_2 = create_step2_prompt(mechanism_b)
    solution_c = loader.prompt(
        prompt_2,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.8,
    ).strip()
    
    print(f"-> Generated Solution (C):\n{solution_c}")
    
    return {
        "object": object_name,
        "mechanism": mechanism_b,
        "solution": solution_c
    }

# --- דוגמא להרצה (שים את זה בקוד שלך) ---
# נניח ש-loader כבר מוגדר אצלך
objects_to_test = ["Rubber Band"]#, "Brick", "Paperclip"]

for obj in objects_to_test:
    run_creative_bridge(loader, obj)


--- Processing: Rubber Band ---
-> Extracted Mechanism (B): A material that has low thermal conductivity due to its high density and insulating properties.
-> Generated Solution (C):
Analysis: The property of having low thermal conductivity allows us to create a device that is highly insulated from heat sources. This means that if we were stranded or trapped in a situation where there was no power available for heating devices, such as heaters or stoves, we could survive by using this device to shield ourselves from heat.

Invention Name: Heat Shield Device
Mechanism: The Heat Shield Device consists of a large, insulated panel that can be placed over a person or area of the body that may come into contact with hot surfaces or equipment. The insulation layer prevents heat from penetrating the device and reaching the user's skin, keeping them cool and comfortable while providing protection against potential burns or injuries from hot objects.

The Heat Shield Device can be used in vario

## Multi-turn chat

Pass a list of messages for back-and-forth conversation.

In [ ]:
messages = [
    {"role": "user", "content": "List 3 uses for a shoelace."},
    {"role": "assistant", "content": "Tie shoes, use as a belt, bundle cables."},
    {"role": "user", "content": "Now give 3 weirder ones."},
]

reply = loader.generate(messages, max_new_tokens=256, do_sample=True, temperature=0.7)
print("Reply:")
print(reply)